In [13]:
import os

os.makedirs("data/raw", exist_ok=True)

In [2]:
import pandas as pd

files = {
    "riverview": "data/raw/riverview.csv",
    "houston_buffalo_bayou": "data/raw/houston.csv",
    "des_moines": "data/raw/des_moines.csv",
    "sacramento": "data/raw/sacramento.csv",
    "baton_rouge": "data/raw/baton_rouge.csv",
    "kansas_city": "data/raw/kansas_city.csv",
    "memphis": "data/raw/memphis.csv",
    "toledo": "data/raw/toledo.csv",
    "st_louis": "data/raw/st_louis.csv",
    "columbus": "data/raw/columbus.csv",
}

dfs = []

for location, path in files.items():
    df = pd.read_csv(path)
    df["location"] = location
    dfs.append(df)

raw_df = pd.concat(dfs, ignore_index=True)

print(raw_df.shape)
raw_df.head()

C:\Users\kcui2\AppData\Local\Temp\ipykernel_28888\3305303501.py:19: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)
C:\Users\kcui2\AppData\Local\Temp\ipykernel_28888\3305303501.py:19: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)
C:\Users\kcui2\AppData\Local\Temp\ipykernel_28888\3305303501.py:19: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)
C:\Users\kcui2\AppData\Local\Temp\ipykernel_28888\3305303501.py:19: DtypeWarning: Columns (8,11) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)
C:\Users\kcui2\AppData\Local\Temp\ipykernel_28888\3305303501.py:19: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


(537157, 19)


C:\Users\kcui2\AppData\Local\Temp\ipykernel_28888\3305303501.py:19: DtypeWarning: Columns (2,3,8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


,ActivityStartDate,ActivityStartTime/Time,ActivityEndDate,ActivityEndTime/Time,MonitoringLocationIdentifier,ActivityDepthHeightMeasure/MeasureValue,ActivityMediaName,CharacteristicName,ResultMeasureValue,ResultMeasure/MeasureUnitCode,ResultStatusIdentifier,MeasureQualifierCode,DetectionQuantitationLimitMeasure/MeasureValue,ResultDetectionConditionText,ProjectIdentifier,ResultAnalyticalMethod/MethodName,ProviderName,ActivityDepthHeightMeasure/MeasureUnitCode,location
0,2016-06-08,08:38:01,NaN,NaN,21FLTBW_WQX-AR510461,0.2,Water,Salinity,0.06,PSS,Final,NaN,NaN,NaN,ATBCHBMP,In Situ Profile,STORET,m,riverview
1,2016-06-28,12:33:00,NaN,NaN,21FLHILL_WQX-178,0.5,Water,"Chlorophyll a, corrected for pheophytin",8.0,ug/L,Final,NaN,13.6,NaN,SWQ,10200 H ~ Chlorophyll a-b-c Determination,STORET,m,riverview
2,2016-10-26,10:59:00,NaN,NaN,21FLHILL_WQX-179,0.5,Water,Enterococcus,1300,#/100mL,Final,NaN,4.0,NaN,SWQ,Enterococci in Water by Membrane Filtration Us...,STORET,m,riverview
3,2016-12-28,09:09:00,NaN,NaN,21FLHILL_WQX-179,0.5,Water,"Nitrogen, mixed forms (NH3), (NH4), organic, (...",1.949,mg/L,Final,NaN,0.116,NaN,SWQ,Total Nitrogen,STORET,m,riverview
4,2016-10-17,12:13:00,NaN,NaN,21FLTBW_WQX-AR305825,0.5,Water,Dissolved oxygen (DO),8.99,mg/L,Final,NaN,NaN,NaN,ATBCHBMP,FDEP FT1500,STORET,m,riverview


In [3]:
global_vars = [
    "pH",
    "Chloride",
    "Specific conductance",
    "Sulfate",
    "Total dissolved solids",
    "Phosphorus"
]

In [4]:
df = raw_df.copy()

df = df[[
    "location",
    "ActivityStartDate",
    "MonitoringLocationIdentifier",
    "CharacteristicName",
    "ResultMeasureValue",
    "ResultMeasure/MeasureUnitCode"
]].copy()

df.head()

,location,ActivityStartDate,MonitoringLocationIdentifier,CharacteristicName,ResultMeasureValue,ResultMeasure/MeasureUnitCode
0,riverview,2016-06-08,21FLTBW_WQX-AR510461,Salinity,0.06,PSS
1,riverview,2016-06-28,21FLHILL_WQX-178,"Chlorophyll a, corrected for pheophytin",8.0,ug/L
2,riverview,2016-10-26,21FLHILL_WQX-179,Enterococcus,1300,#/100mL
3,riverview,2016-12-28,21FLHILL_WQX-179,"Nitrogen, mixed forms (NH3), (NH4), organic, (...",1.949,mg/L
4,riverview,2016-10-17,21FLTBW_WQX-AR305825,Dissolved oxygen (DO),8.99,mg/L


In [5]:
df = df[df["CharacteristicName"].isin(global_vars)].copy()

print(df.shape)
df["CharacteristicName"].value_counts()

(95665, 6)


CharacteristicName
Specific conductance      52116
pH                        25950
Chloride                   5594
Phosphorus                 5329
Sulfate                    3646
Total dissolved solids     3030
Name: count, dtype: int64

In [6]:
df["ActivityStartDate"] = pd.to_datetime(df["ActivityStartDate"], errors="coerce")
df["ResultMeasureValue"] = pd.to_numeric(df["ResultMeasureValue"], errors="coerce")

df = df.dropna(subset=[
    "location",
    "ActivityStartDate",
    "CharacteristicName",
    "ResultMeasureValue"
]).copy()

df = df.rename(columns={
    "ActivityStartDate": "date",
    "CharacteristicName": "characteristic",
    "ResultMeasureValue": "value",
    "ResultMeasure/MeasureUnitCode": "unit",
    "MonitoringLocationIdentifier": "site_id"
})

df.head()

,location,date,site_id,characteristic,value,unit
16,riverview,2016-10-17,21FLTBW_WQX-AR306708,pH,7.82,NaN
28,riverview,2016-04-14,21FLTBW_WQX-AR510218,pH,7.67,NaN
31,riverview,2016-02-01,21FLTBW_WQX-AR509332,Specific conductance,333.00,umho/cm
33,riverview,2016-11-21,21FLTBW_WQX-AR306814,pH,7.45,NaN
38,riverview,2016-09-28,21FLTBW_WQX-AR305825,Specific conductance,330.00,umho/cm


In [7]:
unit_check = (
    df.groupby(["characteristic", "unit"])
      .size()
      .reset_index(name="count")
      .sort_values(["characteristic", "count"], ascending=[True, False])
)

unit_check

,characteristic,unit,count
0,Chloride,mg/L,4617
1,Chloride,mg/l,778
2,Phosphorus,mg/L,3999
5,Phosphorus,mg/l as P,893
6,Phosphorus,ug/L,338
3,Phosphorus,mg/kg,13
4,Phosphorus,mg/kg as P,1
10,Specific conductance,uS/cm,40419
12,Specific conductance,umho/cm,9746
11,Specific conductance,uS/cm @25C,1824


In [8]:
top_units = (
    unit_check.sort_values(["characteristic", "count"], ascending=[True, False])
    .drop_duplicates(subset=["characteristic"])
    .rename(columns={"unit": "top_unit"})
)

df = df.merge(top_units[["characteristic", "top_unit"]], on="characteristic", how="left")
df = df[df["unit"] == df["top_unit"]].copy()
df = df.drop(columns=["top_unit"])

In [10]:
unit_check = (
    df.groupby(["characteristic", "unit"])
      .size()
      .reset_index(name="count")
      .sort_values(["characteristic", "count"], ascending=[True, False])
)

unit_check

,characteristic,unit,count
0,Chloride,mg/L,4617
1,Phosphorus,mg/L,3999
2,Specific conductance,uS/cm,40419
3,Sulfate,mg/L,3121
4,Total dissolved solids,mg/L,1325
5,pH,std units,1219


In [11]:
df["month"] = df["date"].dt.to_period("M").dt.to_timestamp()

monthly_long = (
    df.groupby(["location", "month", "characteristic"], as_index=False)
      .agg(
          monthly_mean=("value", "mean"),
          monthly_median=("value", "median"),
          n_samples=("value", "size")
      )
)

monthly_long.head()

,location,month,characteristic,monthly_mean,monthly_median,n_samples
0,baton_rouge,2010-01-01,pH,7.950,7.95,2
1,baton_rouge,2010-02-01,pH,8.000,8.00,2
2,baton_rouge,2010-03-01,pH,8.000,8.00,4
3,baton_rouge,2010-04-01,pH,7.975,7.95,4
4,baton_rouge,2010-05-01,pH,7.950,7.95,4


In [14]:
os.makedirs("data/processed", exist_ok=True)
monthly_long.to_csv("data/processed/monthly_long_global.csv", index=False)

In [21]:
model_df = (
    monthly_long.pivot_table(
        index=["location", "month"],
        columns="characteristic",
        values="monthly_mean"
    )
    .reset_index()
    .sort_values(["location", "month"])
)

model_df.columns.name = None
model_df.shape

(1029, 8)

In [22]:
model_df.head()

,location,month,Chloride,Phosphorus,Specific conductance,Sulfate,Total dissolved solids,pH
0,baton_rouge,2010-01-01,NaN,NaN,NaN,NaN,NaN,7.950
1,baton_rouge,2010-02-01,NaN,NaN,NaN,NaN,NaN,8.000
2,baton_rouge,2010-03-01,NaN,NaN,NaN,NaN,NaN,8.000
3,baton_rouge,2010-04-01,NaN,NaN,NaN,NaN,NaN,7.975
4,baton_rouge,2010-05-01,NaN,NaN,NaN,NaN,NaN,7.950


In [23]:
model_df.isna().mean()

location                  0.000000
month                     0.000000
Chloride                  0.310010
Phosphorus                0.507289
Specific conductance      0.699708
Sulfate                   0.270165
Total dissolved solids    0.514091
pH                        0.595724
dtype: float64

In [24]:
os.makedirs("data/final", exist_ok=True)
model_df.to_csv("data/final/global_monthly_model_dataset.csv", index=False)